> **Versi publik untuk arsip repositori.** Daftar jurnal kandidat dan peringkat manual dikosongkan; email diganti contoh. Selebihnya sama dengan notebook yang dijalankan pada 27 Oktober 2026.

# Studi A dan C — *Membaca Angka, Menilai Riset* (versi bersih)

* **Studi A (Bab 4)** — cakupan DOI, Crossref, dan OpenAlex untuk artikel jurnal terakreditasi Indonesia.
* **Studi C (Bab 10)** — apakah LLM (Gemini dan ChatGPT) menilai versi Indonesia dan versi Inggris dari artikel yang sama secara setara.

**Cara pakai**
1. *Runtime → Change runtime type → **T4 GPU*** (untuk terjemahan di Langkah 3b).
2. *Runtime → **Run all***.

Notebook ini menyiapkan data (Studi A dan terjemahan untuk Studi C). Penilaian LLM ada di notebook terpisah, `Studi_C_Penilaian_LLM.ipynb`.

Notebook ini **melanjutkan** dari file yang sudah ada di folder Drive `StudiBuku`: jurnal terpilih dan sampel artikel **tidak diundi ulang**, dan terjemahan yang sudah ada dipakai lagi. Bila terputus, cukup *Run all* lagi.

## Langkah 0 — Persiapan

In [ ]:
!pip -q install google-genai openai rapidfuzz langdetect sentencepiece
import os, re, json, time, random, html, hashlib
import requests, pandas as pd
import xml.etree.ElementTree as ET
from rapidfuzz import fuzz
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/StudiBuku'
os.makedirs(OUT, exist_ok=True)
MAILTO = 'email.anda@contoh.ac.id'  # ganti dengan email Anda   # dipakai sebagai identitas sopan untuk API OpenAlex/Crossref
random.seed(42)
print('Folder hasil:', OUT)
FORCE_ULANG = False   # True = undi ulang jurnal & artikel dari nol (JANGAN, kecuali memulai studi baru)
def s_(x): return x if isinstance(x, str) else ''

## Langkah 1 — Daftar kandidat jurnal dan peringkat manual

In [ ]:
CANDIDATES = [
    # (bidang, strata_dugaan, url_jurnal)
    # Versi publik: daftar 47 jurnal kandidat yang dipakai dalam studi tidak didistribusikan,
    # sesuai protokol (hasil hanya dilaporkan agregat, nama jurnal tidak disebut).
    # Isi dengan jurnal OJS Anda sendiri, contoh:
    # ('informatika', 'atas', 'https://contoh-jurnal.ac.id/index.php/jurnal'),
]


In [ ]:
NS = {'oai':'http://www.openarchives.org/OAI/2.0/',
      'oai_dc':'http://www.openarchives.org/OAI/2.0/oai_dc/',
      'dc':'http://purl.org/dc/elements/1.1/'}
XMLLANG = '{http://www.w3.org/XML/1998/namespace}lang'
UA = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36',
      'Accept':'text/xml,application/xml,text/html;q=0.9,*/*;q=0.8'}
from lxml import etree
PARSER = etree.XMLParser(recover=True, huge_tree=True)

def fetch(url, params=None, tries=3):
    for i in range(tries):
        try:
            r = requests.get(url, params=params, timeout=60, headers=UA)
            r.raise_for_status()
            return r
        except Exception as e:
            last = e; time.sleep(3 * (i + 1))
    raise last

def clean(t):
    t = html.unescape(re.sub(r'<[^>]+>', ' ', t or ''))
    return re.sub(r'\s+', ' ', t).strip()

def lang_of(text, tag):
    tag = (tag or '').lower()
    if tag.startswith('id') or tag.startswith('in'): return 'id'
    if tag.startswith('en'): return 'en'
    try:
        l = detect(text)
        return 'id' if l in ('id','ms') else ('en' if l=='en' else l)
    except Exception:
        return None

def harvest(base):
    url = base.rstrip('/') + '/oai'
    params = {'verb':'ListRecords','metadataPrefix':'oai_dc'}
    recs = []
    for _ in range(200):
        r = fetch(url, params)
        content = re.sub(rb'[\x00-\x08\x0b\x0c\x0e-\x1f]', b'', r.content)
        root = etree.fromstring(content, PARSER)
        for rec in root.iter('{%s}record' % NS['oai']):
            md_ = rec.find('.//oai_dc:dc', NS)
            if md_ is None: continue
            get = lambda f: md_.findall('dc:'+f, NS)
            recs.append({
                'oai_id': rec.findtext('.//oai:identifier', namespaces=NS),
                'titles': [(e.get(XMLLANG), clean(e.text)) for e in get('title')],
                'abstracts': [(e.get(XMLLANG), clean(e.text)) for e in get('description')],
                'date': (get('date')[0].text if get('date') else ''),
                'identifiers': [e.text for e in get('identifier') if e.text],
                'source': ' | '.join(e.text for e in get('source') if e.text),
                'rights': ' | '.join(e.text for e in get('rights') if e.text),
                'n_creators': len(get('creator')),
            })
        tok = root.find('.//oai:resumptionToken', NS)
        if tok is None or not (tok.text or '').strip(): break
        params = {'verb':'ListRecords','resumptionToken':tok.text.strip()}
        time.sleep(1)
    return recs

def split_combined(txt):
    # abstrak dua bahasa yang digabung dalam satu kolom: pisahkan di penanda Abstrak/Abstract kedua
    marks = [m.start() for m in re.finditer(r'\b(?:Abstrak|ABSTRAK|Abstract|ABSTRACT)\b', txt)]
    if len(marks) >= 2:
        return [txt[:marks[1]], txt[marks[1]:]]
    return [txt]

def pick_by_lang_ab(pairs):
    out = pick_by_lang(pairs)
    if not ('id' in out and 'en' in out):
        parts = [(None, p) for _, t in pairs for p in split_combined(t or '')]
        alt = pick_by_lang(parts)
        if 'id' in alt and 'en' in alt: out = alt
    return out

def pick_by_lang(pairs):
    out = {}
    for tag, txt in pairs:
        if not txt: continue
        l = lang_of(txt, tag)
        if l in ('id','en') and l not in out: out[l] = txt
    return out

def diagnose(recs):
    n_yr = n_id = n_en = n_dua = 0
    for r in recs:
        yr = re.match(r'(\d{4})', r['date'] or '')
        if not yr or not (YEAR_FROM <= int(yr.group(1)) <= YEAR_TO): continue
        n_yr += 1; ab = pick_by_lang_ab(r['abstracts'])
        n_id += 'id' in ab; n_en += 'en' in ab; n_dua += ('id' in ab and 'en' in ab)
    return {'n_tahun': n_yr, 'n_abstrak_id': n_id, 'n_abstrak_en': n_en, 'n_dua_bahasa': n_dua}

def eligible(recs):
    ok = []
    for r in recs:
        yr = re.match(r'(\d{4})', r['date'] or '')
        if not yr or not (YEAR_FROM <= int(yr.group(1)) <= YEAR_TO): continue
        ab = {k: v for k, v in pick_by_lang_ab(r['abstracts']).items() if len(v.split()) >= MIN_WORDS}
        if not ab: continue                      # minimal satu abstrak (id atau en) yang cukup panjang
        ti = pick_by_lang(r['titles'])
        doi = next((m.group(0) for i in r['identifiers'] for m in [re.search(r'10\.\d{4,9}/\S+', i)] if m), '')
        issn = re.findall(r'\b\d{4}-\d{3}[\dXx]\b', r['source'])
        ok.append({'oai_id':r['oai_id'],'year':int(yr.group(1)),'doi':doi.rstrip('.'),
                   'issn':';'.join(sorted(set(issn))),'title_id':ti.get('id',''),'title_en':ti.get('en',''),
                   'abstract_id':ab.get('id',''),'abstract_en':ab.get('en',''),
                   'bahasa_asli':('dua' if len(ab) == 2 else next(iter(ab))),'n_authors':r['n_creators'],
                   'rights':r['rights'],'source':r['source']})
    return ok

def sinta_rank(base):
    # peringkat SINTA yang tertera di laman jurnal; None bila tidak ada atau ambigu
    try:
        t = fetch(base, tries=2).text
    except Exception:
        return None
    t = html.unescape(re.sub(r'<[^>]+>', ' ', t))
    vals = set()
    for m in re.finditer(r'\b(?:SINTA|Sinta|sinta)\s*-?\s*([1-6])\b|\bS([1-6])\b|[Pp]eringkat\s*([1-6])\b', t):
        k = int(next(g for g in m.groups() if g))
        win = t[max(0, m.start()-80): m.end()+80]
        if m.group(1) or re.search(r'[Aa]kredit|[Aa]ccredit', win):
            vals.add(k)
    return vals.pop() if len(vals) == 1 else None

In [ ]:
MANUAL_RANK = {
    # url_jurnal: peringkat akreditasi bila tidak terbaca otomatis dari laman jurnal.
    # Versi publik: isian studi asli tidak didistribusikan.
}
MANUAL_RANK = {u: k for u, k in MANUAL_RANK.items() if k is not None}
print(len(MANUAL_RANK), 'peringkat manual')

## Langkah 1b — Saring dan pilih jurnal secara acak

Bila `jurnal_terpilih.csv` sudah ada di Drive, langkah ini **hanya memuat** daftar itu (tidak memanen ulang).

In [ ]:
kj_path, jt_path = f'{OUT}/kandidat_jurnal.csv', f'{OUT}/jurnal_terpilih.csv'
CACHE = {}
if os.path.exists(jt_path) and not FORCE_ULANG:
    scr = pd.read_csv(kj_path)
    jt = pd.read_csv(jt_path)
    JOURNALS = [(r.field, int(r['rank']), r.url) for _, r in jt.iterrows()]
    print('Memakai jurnal terpilih yang tersimpan:')
    print(jt.groupby(['field','rank']).size())
else:
    scr = []
    for field, strat, base in CANDIDATES:
        row = {'field':field,'strata_dugaan':strat,'url':base}
        try:
            recs = harvest(base); CACHE[base] = recs
            row['oai_ok'] = True; row['n_rekaman'] = len(recs); row['n_layak'] = len(eligible(recs)); row.update(diagnose(recs))
        except Exception as e:
            row['oai_ok'] = False; row['n_rekaman'] = 0; row['n_layak'] = 0; row['error'] = repr(e)[:120]
        row['sinta_di_laman'] = sinta_rank(base)
        scr.append(row); print(row)
    scr = pd.DataFrame(scr)

    # coba ulang jurnal yang gagal dipanen (server sibuk/menolak sementara)
    for base in scr[~scr.oai_ok].url.tolist():
        time.sleep(10)
        try:
            recs = harvest(base); CACHE[base] = recs
            k = scr.index[scr.url == base][0]
            scr.loc[k, ['oai_ok','n_rekaman','n_layak']] = [True, len(recs), len(eligible(recs))]
            for kk, v in diagnose(recs).items(): scr.loc[k, kk] = v
            if pd.isna(scr.loc[k, 'sinta_di_laman']): scr.loc[k, 'sinta_di_laman'] = sinta_rank(base)
            print('BERHASIL ulang:', base)
        except Exception as e:
            print('tetap gagal:', base)

    def strata_of(r):
        k = r['sinta_di_laman']
        if pd.isna(k): return None            # tidak tertera: cek manual di SINTA
        return 'atas' if k <= 2 else ('bawah' if k <= 4 else 'keluar')
    scr['strata'] = scr.apply(strata_of, axis=1)
    scr.to_csv(f'{OUT}/kandidat_jurnal.csv', index=False)
    cek = scr[(scr.n_layak >= MIN_LAYAK) & (scr.strata.isna())]
    print('\nPERLU CEK PERINGKAT DI SINTA (isi di MANUAL_RANK):')
    for u in cek.url: print('  ', u)
    print('\nRingkasan lolos per kelompok (sebelum cek manual):')
    print(scr[(scr.n_layak >= MIN_LAYAK)].groupby(['field','strata'], dropna=False).size())
    for u, k in MANUAL_RANK.items():
        scr.loc[scr.url == u, 'sinta_di_laman'] = k
    scr['strata'] = scr.apply(strata_of, axis=1)
    scr.to_csv(f'{OUT}/kandidat_jurnal.csv', index=False)
    pool = scr[(scr.oai_ok) & (scr.n_layak >= MIN_LAYAK) & (scr.strata.isin(['atas','bawah']))]
    rng = random.Random(2026)
    sel = []
    for (f, st), g in pool.groupby(['field','strata']):
        urls = sorted(g.url)
        pick = rng.sample(urls, min(N_PER_STRATUM, len(urls)))
        sel += [(f, 2 if st=='atas' else 4, u) for u in pick]
        print(f, st, ':', len(urls), 'lolos, dipilih', len(pick))
    JOURNALS = sel
    jt = pd.DataFrame(JOURNALS, columns=['field','rank','url'])
    jt.to_csv(f'{OUT}/jurnal_terpilih.csv', index=False)
    kurang = jt.groupby(['field','rank']).size()
    print(kurang)
    assert len(kurang) == 4 and kurang.min() >= 3, 'Ada kelompok dengan kurang dari 3 jurnal: kirim hasilnya ke saya.'

## Langkah 2 — Sampel artikel

Bila `articles.csv` sudah ada, sampel itu dipakai apa adanya. Rekaman ganda (satu jurnal mengirim artikel yang sama dua kali) dibuang.

In [ ]:
art_path = f'{OUT}/articles.csv'
if os.path.exists(art_path) and not FORCE_ULANG:
    art = pd.read_csv(art_path, keep_default_na=False)
    n0 = len(art)
    art = art.drop_duplicates('article_id').reset_index(drop=True)
    if len(art) < n0:
        print('Rekaman ganda dibuang:', n0 - len(art))
        art.to_csv(art_path, index=False)
else:
    def quotas(caps, total):
        # bagi rata 'total' artikel ke jurnal-jurnal; sisa kuota dialihkan ke jurnal yang masih punya artikel
        q = {u: 0 for u in caps}
        while sum(q.values()) < total:
            open_ = [u for u in sorted(caps) if q[u] < caps[u]]
            if not open_: break
            for u in open_:
                if sum(q.values()) < total and q[u] < caps[u]: q[u] += 1
        return q

    POOL = {b: eligible(CACHE.get(b) or harvest(b)) for _, _, b in JOURNALS}
    rows = []
    for (field, rank), g in pd.DataFrame(JOURNALS, columns=['field','rank','url']).groupby(['field','rank']):
        q = quotas({u: len(POOL[u]) for u in g.url}, N_PER_GROUP)
        for base, k in q.items():
            ok = list(POOL[base]); random.Random(42).shuffle(ok)
            seen = set(); ok = [d for d in ok if not (d['oai_id'] in seen or seen.add(d['oai_id']))]   # buang rekaman ganda
            for d in ok[:k]:
                d.update({'field':field,'rank':rank,'journal_url':base}); rows.append(d)
            print(f'{base}: diambil {k}')
    art = pd.DataFrame(rows)
    art.insert(0, 'article_id', [hashlib.md5(x.encode()).hexdigest()[:10] for x in art['oai_id']])
    art = art.drop_duplicates('article_id').reset_index(drop=True)
    art.to_csv(art_path, index=False)

art['year'] = art['year'].astype(int)
assert art.article_id.is_unique
print('Total artikel sampel:', len(art))
print(art.groupby(['field','rank']).size())

## Langkah 3 — Studi A: DOI, Crossref, dan OpenAlex

In [ ]:
def openalex_by_doi(doi):
    r = requests.get(f'https://api.openalex.org/works/doi:{doi}', params={'mailto':MAILTO}, timeout=30)
    return r.json() if r.status_code == 200 else None

def openalex_by_title(title, year):
    if not title: return None
    q = re.sub(r'[^\w\s]', ' ', title)[:200]
    r = requests.get('https://api.openalex.org/works',
                     params={'filter':f'title.search:{q},publication_year:{year-1}-{year+1}','per-page':5,'mailto':MAILTO}, timeout=30)
    if r.status_code != 200: return None
    for w in r.json().get('results', []):
        if fuzz.token_set_ratio(title.lower(), (w.get('title') or '').lower()) >= 90: return w
    return None

def crossref_ok(doi):
    if not doi: return False
    r = requests.get(f'https://api.crossref.org/works/{doi}', params={'mailto':MAILTO}, timeout=30)
    return r.status_code == 200

cov_path = f'{OUT}/coverage.csv'
done = set(pd.read_csv(cov_path)['article_id']) if os.path.exists(cov_path) else set()
out = []
for _, a in art.iterrows():
    if a.article_id in done: continue
    w, how = None, ''
    if a.doi: w = openalex_by_doi(a.doi); how = 'doi' if w else ''
    if w is None:
        w = openalex_by_title(a.title_en or a.title_id, a.year); how = 'judul' if w else ''
    out.append({'article_id':a.article_id, 'has_doi':bool(a.doi), 'crossref':crossref_ok(a.doi),
                'openalex':w is not None, 'match_by':how,
                'openalex_id':(w or {}).get('id',''), 'cited_by':(w or {}).get('cited_by_count', None),
                'oa_language':(w or {}).get('language',''),
                'oa_source':(((w or {}).get('primary_location') or {}).get('source') or {}).get('display_name','')})
    if len(out) % 20 == 0:
        pd.DataFrame(out).to_csv(cov_path, mode='a', header=not os.path.exists(cov_path), index=False); out = []
    time.sleep(0.2)
if out: pd.DataFrame(out).to_csv(cov_path, mode='a', header=not os.path.exists(cov_path), index=False)
cov = pd.read_csv(cov_path).drop_duplicates('article_id', keep='last').merge(art[['article_id','field','rank']], on='article_id')
cov.groupby(['field','rank'])[['has_doi','crossref','openalex']].mean().round(3)

In [ ]:
# verifikasi DOI yang tidak dikenali Crossref, dan DOI dari OpenAlex untuk artikel tanpa DOI di metadata
chk_path = f'{OUT}/doi_check.csv'
cov = pd.read_csv(cov_path).drop_duplicates('article_id', keep='last').merge(art[['article_id','doi','journal_url']], on='article_id')
old_chk = pd.read_csv(chk_path) if os.path.exists(chk_path) else pd.DataFrame(columns=['article_id'])
rows = []
for _, r in cov[(cov.has_doi) & (~cov.crossref) & (~cov.article_id.isin(old_chk.article_id))].iterrows():
    doi = r.doi
    cr = requests.get(f'https://api.crossref.org/works/{doi}', params={'mailto':MAILTO}, timeout=30).status_code
    ag = requests.get(f'https://api.crossref.org/works/{doi}/agency', params={'mailto':MAILTO}, timeout=30)
    agency = ag.json()['message']['agency']['id'] if ag.status_code == 200 else ''
    hd = requests.get(f'https://doi.org/api/handles/{doi}', timeout=30)
    rows.append({'article_id':r.article_id, 'jenis':'doi_tak_dikenal_crossref', 'doi':doi,
                 'crossref_ulang':cr, 'agensi':agency, 'doi_terdaftar':hd.status_code == 200})
    time.sleep(1)
for _, r in cov[(~cov.has_doi) & (cov.openalex) & (~cov.article_id.isin(old_chk.article_id))].iterrows():
    w = requests.get('https://api.openalex.org/works/' + r.openalex_id.split('/')[-1], params={'mailto':MAILTO}, timeout=30)
    rows.append({'article_id':r.article_id, 'jenis':'tanpa_doi_di_metadata',
                 'doi':((w.json().get('doi') or '') if w.status_code == 200 else '')})
    time.sleep(0.3)
chk = pd.concat([old_chk, pd.DataFrame(rows)], ignore_index=True)
chk = chk[chk.article_id.isin(art.article_id)]
chk.to_csv(chk_path, index=False)
print(chk.groupby('jenis').size())

## Langkah 3b — Studi C: terjemahan terkendali (butuh GPU)

* Teks penulis ditempatkan menurut **bahasa sebenarnya** (dideteksi otomatis), bukan label metadata, karena sebagian jurnal salah memberi label.
* Versi bahasa yang tidak ditulis penulis dibuat dengan **NLLB-200** (`facebook/nllb-200-distilled-600M`), bukan model yang diuji.
* Setiap hasil terjemahan disimpan di `translations_cache.csv`, sehingga tidak pernah diterjemahkan dua kali. Terjemahan lama yang sah dari `translations.csv` dipakai lagi.
* Hasil akhir: `translations_final.csv`.

In [ ]:
import torch, logging
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
MT_NAME = 'facebook/nllb-200-distilled-600M'
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Perangkat:', dev, '' if dev == 'cuda' else '— PERINGATAN: tanpa GPU, terjemahan sangat lambat')
tok = AutoTokenizer.from_pretrained(MT_NAME)
mt = AutoModelForSeq2SeqLM.from_pretrained(MT_NAME).to(dev).eval()
CODE = {'id':'ind_Latn', 'en':'eng_Latn'}

def split_sent(t): return [x for x in re.split(r'(?<=[.!?])\s+', t) if x.strip()]

def nllb(text, src, tgt):
    sents, out = split_sent(text), []
    tok.src_lang = CODE[src]
    for i in range(0, len(sents), 8):
        b = tok(sents[i:i+8], return_tensors='pt', padding=True, truncation=True, max_length=256).to(dev)
        with torch.no_grad():
            g = mt.generate(**b, forced_bos_token_id=tok.convert_tokens_to_ids(CODE[tgt]),
                            max_new_tokens=256, num_beams=4)
        out += tok.batch_decode(g, skip_special_tokens=True)
    return ' '.join(out)

# --- cache terjemahan ---
cache_path = f'{OUT}/translations_cache.csv'
key = lambda text, src, tgt: hashlib.md5(f'{src}|{tgt}|{text}'.encode()).hexdigest()
CACHE_TR = {}
if os.path.exists(cache_path):
    CACHE_TR = dict(pd.read_csv(cache_path, keep_default_na=False)[['key','out']].values)
elif os.path.exists(f'{OUT}/translations.csv'):           # pakai ulang terjemahan sah dari percobaan sebelumnya
    old = pd.read_csv(f'{OUT}/translations.csv', keep_default_na=False).drop_duplicates('article_id', keep='last')
    for _, r in old.iterrows():
        for lang, other in (('id','en'), ('en','id')):
            if r.get(f'abstract_{lang}_sumber') == 'mesin' and r[f'text_abstract_{other}']:
                CACHE_TR[key(r[f'text_abstract_{other}'], other, lang)] = r[f'text_abstract_{lang}']
                if r[f'text_title_{other}'] and r[f'text_title_{other}'] != r[f'text_title_{lang}']:
                    CACHE_TR[key(r[f'text_title_{other}'], other, lang)] = r[f'text_title_{lang}']
    pd.DataFrame(list(CACHE_TR.items()), columns=['key','out']).to_csv(cache_path, index=False)
print('Terjemahan tersimpan di cache:', len(CACHE_TR))

N_NEW = 0
def translate(text, src, tgt):
    global N_NEW
    if not text: return ''
    k = key(text, src, tgt)
    if k not in CACHE_TR:
        CACHE_TR[k] = nllb(text, src, tgt); N_NEW += 1
        pd.DataFrame([[k, CACHE_TR[k]]], columns=['key','out']).to_csv(cache_path, mode='a', header=False, index=False)
    return CACHE_TR[k]

# --- deteksi bahasa ---
def L(x):
    try: return detect(x) if isinstance(x, str) and len(x.split()) >= 3 else ''
    except Exception: return ''
def lang_ok(x):
    l = L(x); return 'id' if l in ('id','ms') else ('en' if l == 'en' else None)

rows = []
for n, (_, a) in enumerate(art.iterrows(), 1):
    ab, ti = {}, {}
    for slot in ('id', 'en'):
        x = s_(a[f'abstract_{slot}']); lg = lang_ok(x)
        if x and lg and lg not in ab: ab[lg] = x
        y = s_(a[f'title_{slot}']); lt = lang_ok(y) or slot
        if y and lt not in ti: ti[lt] = y
    assert ab, f'Abstrak tidak terbaca: {a.article_id}'
    d = {'article_id': a.article_id, 'penerjemah': MT_NAME}
    for lang, other in (('id', 'en'), ('en', 'id')):
        if lang in ab:
            d[f'abstract_{lang}_sumber'] = 'penulis'; d[f'text_abstract_{lang}'] = ab[lang]
        else:
            d[f'abstract_{lang}_sumber'] = 'mesin'; d[f'text_abstract_{lang}'] = translate(ab[other], other, lang)
        d[f'text_title_{lang}'] = ti[lang] if lang in ti else translate(ti.get(other, ''), other, lang)
    rows.append(d)
    if n % 25 == 0: print(n, '/', len(art), '| terjemahan baru:', N_NEW)

txt = pd.DataFrame(rows)
txt['lang_teks_id'] = txt.text_abstract_id.map(L); txt['lang_teks_en'] = txt.text_abstract_en.map(L)
txt['bahasa_asli_v'] = [('dua' if a == b == 'penulis' else ('id' if a == 'penulis' else 'en'))
                        for a, b in zip(txt.abstract_id_sumber, txt.abstract_en_sumber)]
txt.to_csv(f'{OUT}/translations_final.csv', index=False)
print('Terjemahan baru dibuat:', N_NEW)
print('Bahasa teks (baris = versi ID, kolom = versi EN):'); print(pd.crosstab(txt.lang_teks_id, txt.lang_teks_en))
print(txt.bahasa_asli_v.value_counts())
assert len(txt) == len(art) and (txt.text_abstract_id != txt.text_abstract_en).all()

In [ ]:
# contoh hasil untuk diperiksa
for b in ('en', 'id'):
    s = txt[txt.bahasa_asli_v == b]
    if len(s) == 0: continue
    r = s.sample(1, random_state=1).iloc[0]
    print(f'[asli {b}] ID:', r.text_abstract_id[:300]); print(f'[asli {b}] EN:', r.text_abstract_en[:300]); print('-'*60)

## Selesai — lanjut ke notebook kedua

Persiapan data selesai. Penilaian LLM dijalankan di notebook **`Studi_C_Penilaian_LLM.ipynb`** (tanpa GPU). Runtime ini boleh diputus: *Runtime → Disconnect and delete runtime*.